In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# install stuff for unzipping LLM
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 124 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (742 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10

In [8]:

# 1. Cài đặt các công cụ dò tìm phần cứng (lspci, lshw)
!sudo apt-get update && apt-get install -y pciutils lshw

# 2. Cài đặt Ollama (nếu chưa cài)
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Chạy ngầm Ollama server và ép sử dụng 2 GPU T4

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.

In [9]:
# 1. Cài đặt các gói phụ thuộc dự án (Không cần vLLM giúp tăng tốc độ cài đặt và tránh xung đột)
print("📥 Đang cài đặt dependencies...")
!pip install -q \
    langgraph>=0.2.0 \
    langchain-core>=0.3.0 \
    langchain-openai>=0.2.0 \
    pyyaml>=6.0 \
    json-repair>=0.30.0 \
    openpyxl>=3.1.0 \
    tabulate>=0.9.0 \
    thefuzz>=0.22.0


📥 Đang cài đặt dependencies...
📥 Đang tải và cài đặt Ollama...


In [12]:
%cd /kaggle/working
!git clone https://github.com/Djuybu/r2AI_2026


/kaggle/working
fatal: destination path 'r2AI_2026' already exists and is not an empty directory.


In [15]:
import subprocess
import time
import requests
import os

# 1. Khởi động Ollama Server chạy nền
print("🚀 Đang khởi động Ollama Server...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 2. Chờ Ollama Server sẵn sàng
print("⏳ Chờ Ollama Server khởi động...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/")
        if r.status_code == 200:
            print("✅ Ollama Server đã sẵn sàng tại port 11434!")
            break
    except Exception:
        time.sleep(1)
else:
    print("❌ Lỗi: Ollama Server không thể khởi động.")

# 3. Pull model (Mặc định sử dụng qwen2.5-coder:1.5b gọn nhẹ để demo nhanh)
MODEL_NAME = "qwen2.5-coder:1.5b"
print(f"📥 Đang tải mô hình {MODEL_NAME} từ Ollama registry...")
subprocess.run(["ollama", "pull", MODEL_NAME])
print(f"✅ Đã tải thành công mô hình {MODEL_NAME}!")

🚀 Đang khởi động Ollama Server...
⏳ Chờ Ollama Server khởi động...
[GIN] 2026/08/11 - 09:45:12 | 200 |     131.576µs |       127.0.0.1 | GET      "/"
✅ Ollama Server đã sẵn sàng tại port 11434!
📥 Đang tải mô hình qwen2.5-coder:1.5b từ Ollama registry...
[GIN] 2026/08/11 - 09:45:12 | 200 |      39.813µs |       127.0.0.1 | HEAD     "/"


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ time=2026-08-11T09:45:13.220Z level=INFO source=download.go:181 msg="downloading 29d8c98fa6b0 in 10 100 MB part(s)"
pulling manifest ⠸ pulling manifest 
pulling 29d8c98fa6b0:   6% ▕                  ▏  54 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:   9% ▕█                 ▏  93 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  18% ▕███               ▏ 177 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  26% ▕████              ▏ 261 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  31% ▕█████             ▏ 302 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  39% ▕███████           ▏ 384 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  47% ▕████████          ▏ 467 MB/986 MB                  pulling manifest 
pulling 29d8c98fa6b0:  51% ▕█████████         ▏ 506 MB/986 MB                  pulling manifest 
pulling 29d8c9

[GIN] 2026/08/11 - 09:45:18 | 200 |  5.972907562s |       127.0.0.1 | POST     "/api/pull"
✅ Đã tải thành công mô hình qwen2.5-coder:1.5b!


pulling manifest 
pulling 29d8c98fa6b0:  99% ▕█████████████████ ▏ 975 MB/986 MB  692 MB/s      0s
verifying sha256 digest ⠏ pulling manifest 
pulling 29d8c98fa6b0:  99% ▕█████████████████ ▏ 975 MB/986 MB  692 MB/s      0s
verifying sha256 digest ⠋ pulling manifest 
pulling 29d8c98fa6b0:  99% ▕█████████████████ ▏ 975 MB/986 MB  692 MB/s      0s
verifying sha256 digest 
writing manifest 
success 


In [16]:
# 3. Thiết lập Biến Môi trường & Chạy Thử Kết nối (Sanity Check)
os.environ["MODEL_NAME"] = MODEL_NAME
os.environ["LLM_API_BASE"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"] = "ollama"

from langchain_openai import ChatOpenAI

# Gọi thử model để xác minh hoạt động
llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_base="http://localhost:11434/v1",
    openai_api_key="ollama",
    temperature=0,
)

response = llm.invoke("Xin chào! Kiểm tra kết nối từ LangChain?")
print("🤖 Phản hồi thử nghiệm từ Ollama:")
print(response.content)

time=2026-08-11T09:46:01.379Z level=INFO source=sched.go:1013 msg="selecting single GPU for llama-server model" main_gpu=0 id=0 filter_id=0 library=CUDA name=CUDA0 description="Tesla T4" integrated=false predicted="1.8 GiB" available="14.5 GiB"
time=2026-08-11T09:46:01.379Z level=INFO source=sched.go:1132 msg="selecting GPU backend for llama-server model" library=CUDA gpu_count=1 available_gpu_count=2
time=2026-08-11T09:46:01.379Z level=INFO source=server.go:109 msg="using llama-server for model" model=/root/.ollama/models/blobs/sha256-29d8c98fa6b098e200069bfb88b9508dc3e85586d20cba59f8dda9a808165104
time=2026-08-11T09:46:01.380Z level=INFO source=llama_server.go:431 msg="starting llama-server" cmd="/usr/local/lib/ollama/llama-server --model /root/.ollama/models/blobs/sha256-29d8c98fa6b098e200069bfb88b9508dc3e85586d20cba59f8dda9a808165104 --port 37117 --host 127.0.0.1 --no-webui --offline -c 32768 -np 1 --log-verbosity 4 --no-log-prefix --no-log-timestamps --no-jinja --chat-template cha

[GIN] 2026/08/11 - 09:47:21 | 200 |         1m22s |       127.0.0.1 | POST     "/v1/chat/completions"
🤖 Phản hồi thử nghiệm từ Ollama:
Xin chào! Tôi có thể kiểm tra kết nối của bạn với LangChain bằng cách sử dụng một số lệnh Python. Bạn cần đảm bảo rằng bạn đã cài đặt LangChain và các phụ thuộc cần thiết. Nếu bạn gặp lỗi, hãy thử cung cấp thêm thông tin về lỗi để tôi giúp đỡ hơn.

Ví dụ, nếu bạn muốn kiểm tra kết nối đến một API của LangChain, bạn có thể sử dụng lệnh sau:

```python
import langchain

try:
    response = langchain.llm("Hello, how are you?")
    print(response)
except Exception as e:
    print(f"Error: {e}")
```

Nếu bạn gặp lỗi, hãy cung cấp thêm thông tin về lỗi để tôi giúp đỡ hơn.


slot print_timing: id  0 | task 0 | prompt eval time =   34720.16 ms /    42 tokens (  826.67 ms per token,     1.21 tokens per second)
slot print_timing: id  0 | task 0 |        eval time =   10479.85 ms /   156 tokens (   67.18 ms per token,    14.89 tokens per second)
slot print_timing: id  0 | task 0 |       total time =   45200.01 ms /   198 tokens
slot print_timing: id  0 | task 0 |    graphs reused =        155
slot      release: id  0 | task 0 | stop processing: n_tokens = 197, truncated = 0
srv  update_slots: all slots are idle


In [ ]:
# 1. Dò tìm và liên kết dữ liệu từ Kaggle dataset /kaggle/input/datasets/duymcminh/r2-ai-output
import os
import shutil
from pathlib import Path

# Tìm thư mục dataset chứa qdrant_local_db và ViFinQA
dataset_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'qdrant_local_db' in dirs:
        dataset_path = Path(root)
        break

if not dataset_path:
    fallback_path = Path('/kaggle/input/r2-ai-output')
    if fallback_path.exists():
        dataset_path = fallback_path

if dataset_path:
    print(f"✅ Đã tìm thấy dataset tại: {dataset_path}")
    
    # Tạo symlinks vào repo để RAG module và Agent có thể truy cập trực tiếp
    repo_dir = Path("/kaggle/working/r2AI_2026")
    if repo_dir.exists():
        vifinqa_src = dataset_path / "ViFinQA"
        qdrant_src = dataset_path / "qdrant_local_db"
        bm25_src = dataset_path / "bm25_index.pkl"

        vifinqa_dst = repo_dir / "ViFinQA"
        qdrant_dst = repo_dir / "rag_module" / "qdrant_local_db"
        bm25_dst = repo_dir / "rag_module" / "bm25_index.pkl"

        for src, dst in [(vifinqa_src, vifinqa_dst), (qdrant_src, qdrant_dst), (bm25_src, bm25_dst)]:
            if dst.exists() or dst.is_symlink():
                if dst.is_symlink():
                    os.remove(dst)
                elif dst.is_dir():
                    shutil.rmtree(dst)
                else:
                    os.remove(dst)
            
            if src.exists():
                os.symlink(src, dst)
                print(f"✅ Đã tạo liên kết tượng trưng: {dst} -> {src}")
            else:
                print(f"⚠️ Không tìm thấy nguồn dữ liệu: {src}")
else:
    print("❌ Lỗi: Không tìm thấy thư mục dataset chứa qdrant_local_db.")


In [ ]:
# 2. Chuyển thư mục làm việc vào repository và tải danh sách câu hỏi
import sys
import json

repo_dir = Path("/kaggle/working/r2AI_2026")
os.chdir(str(repo_dir))
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

# Đọc danh sách câu hỏi kiểm thử
questions_file = repo_dir / "ViFinQA" / "questions" / "questions.jsonl"
questions = []
if questions_file.exists():
    with open(questions_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                questions.append(json.loads(line))
    print(f"✅ Đã tải {len(questions)} câu hỏi từ file {questions_file.name}")
else:
    print(f"❌ Không tìm thấy file câu hỏi tại {questions_file}")


In [ ]:
# 3. Khởi tạo Agent và chạy thử nghiệm ngẫu nhiên với 10 câu hỏi
import random
from pipeline.src.graph import create_cocopila_graph

# Khởi tạo ứng dụng LangGraph
print("🤖 Đang khởi tạo Agent workflow...")
app = create_cocopila_graph()

# Lấy ngẫu nhiên 10 câu hỏi kiểm thử
random.seed(42)
if len(questions) >= 10:
    test_questions = random.sample(questions, 10)
else:
    test_questions = questions

print(f"🚀 Bắt đầu kiểm thử với {len(test_questions)} câu hỏi ngẫu nhiên...")
print("=" * 80)

results_summary = []

for idx, q_item in enumerate(test_questions, 1):
    q_id = q_item.get("id")
    q_text = q_item.get("question")
    print(f"\n[{idx}/10] ❓ Câu hỏi ID {q_id}: {q_text}")
    print("-" * 60)
    
    inputs = {
        "user_query": q_text,
        "retry_count": 0,
        "node_latencies": {},
        "status": "pending"
    }
    
    try:
        final_state = None
        for output in app.stream(inputs):
            for node_name, node_state in output.items():
                latency = node_state.get('node_latencies', {}).get(node_name, 'N/A')
                print(f"   📍 Node: [{node_name.upper()}] (Thời gian chạy: {latency}s)")
                if node_name == "executor":
                    final_state = node_state
        
        status = final_state.get("status") if final_state else "error"
        exec_res = final_state.get("execution_result") if final_state else None
        err_msg = final_state.get("error_message") if final_state else None
        err_tb = final_state.get("error_traceback") if final_state else None
        
        print(f"   🏁 Trạng thái kết thúc: {status.upper()}")
        if status == "success":
            data_preview = exec_res.get("data")
            print(f"   ✅ Thành công! Dữ liệu kết quả:")
            print(f"      {json.dumps(data_preview, indent=6, ensure_ascii=False)[:300]}...")
            results_summary.append({
                "id": q_id,
                "question": q_text,
                "status": "success"
            })
        else: 
            print(f"   ❌ Thất bại! Chi tiết lỗi: {err_msg or err_tb or 'Lỗi không xác định'}")
            results_summary.append({
                "id": q_id,
                "question": q_text,
                "status": "failed",
                "error": err_msg or err_tb or "Lỗi không xác định"
            })
            
    except Exception as e:
        print(f"   💥 Lỗi ngoại lệ trong quá trình chạy: {e}")
        results_summary.append({
            "id": q_id,
            "question": q_text,
            "status": "error",
            "error": str(e)
        })

print("\n" + "=" * 80)
print("📊 TỔNG HỢP KẾT QUẢ KIỂM THỬ:")
print("=" * 80)
success_count = sum(1 for r in results_summary if r["status"] == "success")
print(f"Tổng số câu hỏi: {len(results_summary)} | Thành công: {success_count} | Thất bại: {len(results_summary) - success_count}")
for r in results_summary:
    status_emoji = "✅" if r["status"] == "success" else "❌"
    print(f"{status_emoji} ID {r['id']}: {r['status'].upper()}")
